# Try your multi-task model
One T5-small that both removes hostility and fixes register. The **task prefix** picks which job it does. **This notebook does not train anything.**

Run this locally in VS Code using the project's `.venv` Python kernel. No NVIDIA GPU is needed.

Trained by `train_both.ipynb`. For the single-task models, use `inference_detox.ipynb` or `inference_formal.ipynb` instead.

## 1. Set up your local environment
Dependencies are managed in `pyproject.toml`. From a terminal in the project directory, run:

```bash
uv sync
```

Then select **Select Kernel - Python Environments - .venv/bin/python** in VS Code. If it is not listed, use **Enter interpreter path** and choose that file. Run the cell below to confirm the selected interpreter.

In [16]:
import sys
print("Python:", sys.executable)
print("Version:", sys.version.split()[0])

Python: /Users/ayman/projects/second-thought/.venv/bin/python
Version: 3.12.13


## 2. Choose your saved model
The default is `models/both-t5`, matching the `EXPORT_DIR` written by `train_both.ipynb`.

Paths resolve from the project root, found by walking up to `pyproject.toml`. That matters because VS Code starts the kernel in the **notebook's** folder by default, so a bare relative path like `models/...` would resolve to `notebooks/models/...` and fail.

If the folder does not exist but its ZIP sits beside it, this cell extracts the ZIP - which requires the ZIP's top-level folder name to match `MODEL_DIR`'s name. An existing folder is never overwritten.

`MAX_INPUT_TOKENS` is 128, matching the `MAX_LENGTH` the model was trained with. Oversized input is **rejected rather than truncated**, so no text is ever silently dropped.

In [17]:
from pathlib import Path
import zipfile

# Find the project root by walking up to pyproject.toml, so the paths below work
# whichever directory the kernel starts in (VS Code defaults to the notebook's folder).
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
MODEL_DIR = PROJECT_ROOT / "models/both-t5"  # Matches EXPORT_DIR in train_both.ipynb.
DETOX_PREFIX = "detoxify: "    # Must match the training prefixes exactly.
FORMAL_PREFIX = "formalize: "
MAX_INPUT_TOKENS = 128
MAX_NEW_TOKENS = 128
NUM_BEAMS = 4

MODEL_DIR = MODEL_DIR.expanduser().resolve()
archive_path = MODEL_DIR.with_suffix(".zip")
if not MODEL_DIR.exists():
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Cannot find {MODEL_DIR} or {archive_path}. "
            "Choose the path to your extracted local model folder."
        )
    with zipfile.ZipFile(archive_path) as archive:
        # Check paths before extracting the model folder.
        for member in archive.infolist():
            destination = (MODEL_DIR.parent / member.filename).resolve()
            if not destination.is_relative_to(MODEL_DIR):
                raise ValueError(f"Unexpected ZIP member: {member.filename}")
        archive.extractall(MODEL_DIR.parent)
    print("Extracted:", archive_path.name)

required = ["config.json", "model.safetensors", "tokenizer_config.json", "tokenizer.json"]
missing = [name for name in required if not (MODEL_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete model folder {MODEL_DIR}: missing {missing}")
print("Model folder:", MODEL_DIR)

Extracted: both-t5.zip
Model folder: /Users/ayman/projects/second-thought/models/both-t5


## 3. Load the model once
Inference uses your local CPU. Loading happens once; subsequent calls reuse the model. One set of weights serves both tasks.

In [18]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = torch.device("cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR, local_files_only=True)
model.to(device)
model.eval()
print(f"Ready! Using {device}.")

Ready! Using cpu.


## 4. Define the rewriting functions
`rewrite()` takes the prefix explicitly, because one model now answers to two. `detoxify()` and `formalize()` are thin wrappers over it.

The prefixes are **learned task tokens**, not instructions the model understands in general - `"detoxify: "` and `"formalize: "` work because those exact strings were used in training. Changing a character, including the trailing space, produces nonsense.

Beam search runs without random sampling. Blank input returns an empty string; input over 128 tokens is rejected so text is not silently dropped.

In [19]:
@torch.inference_mode()
def rewrite(prefix, text):
    """Rewrite one sentence with the saved model, under the given task prefix."""
    if not isinstance(text, str):
        raise TypeError("Please provide a string.")
    text = text.strip()
    if not text:
        return ""

    inputs = tokenizer(prefix + text, return_tensors="pt", truncation=False)
    if inputs["input_ids"].shape[1] > MAX_INPUT_TOKENS:
        raise ValueError("That text is too long. Please try a shorter sentence.")
    inputs = inputs.to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        do_sample=False,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()


def detoxify(text):
    return rewrite(DETOX_PREFIX, text)


def formalize(text):
    return rewrite(FORMAL_PREFIX, text)

## 5. Check the prefixes still mean different things
**Run this before trusting any other output in this notebook.**

Multi-task training only works if the model learned that the prefix selects the behaviour. When it fails - which is what happens if the two corpora were trained in sequence instead of mixed, or if the training shuffle was skipped - the prefixes **collapse**: both return the same text, and the model has quietly become a single-task model wearing two labels.

Each probe below is both hostile and informal, so a working model should treat them visibly differently: `detoxify:` should soften the insult and leave the casual wording alone, `formalize:` should fix the wording and may leave the insult.

In [20]:
PROBES = [
    "this is the dumbest plan ive heard tbh",
    "ur code is garbage and u clearly didnt test it",
    "cant believe u screwed this up again, thats so lazy",
    "whoever wrote this spec is an idiot, its completely useless",
]

differ = 0
for text in PROBES:
    detox_out, formal_out = detoxify(text), formalize(text)
    same = detox_out == formal_out
    differ += not same
    print("INPUT     :", text)
    print("detoxify: :", detox_out)
    print("formalize::", formal_out)
    print("          ", "same output <- prefixes collapsed here" if same else "different - prefix is doing work")
    print()

print(f"{differ} of {len(PROBES)} probes gave different output per prefix.")
if differ == 0:
    print("The prefixes have collapsed. This model is not behaving as a multi-task model.")
elif differ < len(PROBES):
    print("Partial separation: the prefix works on some inputs but not all.")
else:
    print("Full separation: the prefix selects the behaviour on every probe.")

INPUT     : this is the dumbest plan ive heard tbh
detoxify: : this is the worst plan i have heard tbh
formalize:: This is the worst plan I have heard.
           different - prefix is doing work

INPUT     : ur code is garbage and u clearly didnt test it
detoxify: : ur code is bad and u clearly didnt test it
formalize:: Your code is garbage and you clearly did not test it.
           different - prefix is doing work

INPUT     : cant believe u screwed this up again, thats so lazy
detoxify: : cant believe u screwed this up again, thats so lazy
formalize:: I cannot believe you screwed this up again, that is so lazy.
           different - prefix is doing work

INPUT     : whoever wrote this spec is an idiot, its completely useless
detoxify: : whoever wrote this spec is not good, its completely useless
formalize:: Who wrote this spec is an idiot, it is completely useless.
           different - prefix is doing work

4 of 4 probes gave different output per prefix.
Full separation: the pre

## 6. Each task on its own
Inputs that exercise one axis at a time: hostile but already well-formed, and casual but perfectly polite. A good multi-task model handles each without needing the other.

In [21]:
TOXIC = [
    "this is a stupid idea and you never listen",
    "that was a moronic decision by management",
    "your refactor is trash, revert it",
]
INFORMAL = [
    "gonna need that report asap",
    "i dont think thats gonna work tbh",
    "hey can u send me the deck before the meeting",
]

print("=" * 72)
print("detoxify:")
print("=" * 72)
for text in TOXIC:
    print("  IN :", text)
    print("  OUT:", detoxify(text))
    print()

print("=" * 72)
print("formalize:")
print("=" * 72)
for text in INFORMAL:
    print("  IN :", text)
    print("  OUT:", formalize(text))
    print()

detoxify:
  IN : this is a stupid idea and you never listen
  OUT: this is a bad idea and you never listen

  IN : that was a moronic decision by management
  OUT: that was a bad decision by management

  IN : your refactor is trash, revert it
  OUT: your refactor is bad, revert it

formalize:
  IN : gonna need that report asap
  OUT: I will need that report as soon as possible.

  IN : i dont think thats gonna work tbh
  OUT: I do not think that will work.

  IN : hey can u send me the deck before the meeting
  OUT: Hello, can you send me the deck before the meeting?



## 7. Chain both tasks
A message can be hostile **and** too casual at once. Because one model does both, you can pass its own output back in under the other prefix.

Order matters, and neither is obviously right: detoxifying first means the formalizer never sees the insult, while formalizing first gives the detoxifier cleaner input to work with. The cell runs both orders so you can compare on your own text.

This is also the honest limit of a single model - it does one job per pass. Two passes, two prefixes.

In [25]:
MIXED = [
    "Im gonna work on an assignment tomorrow, i need the fucking report asap by eod",
    "Who the hell sent the boxes to 47 Willow Lane? They were supposed to go to 92 Cedar Court.",
    "Stop being such a jerk to Nina. She's covering 3 shifts this week because you bailed.",
]

for text in MIXED:
    detox_first = formalize(detoxify(text))
    formal_first = detoxify(formalize(text))
    print("INPUT               :", text)
    print("detoxify > formalize:", detox_first)
    print("formalize > detoxify:", formal_first)
    print("orders agree        :", detox_first == formal_first)
    print()

INPUT               : Im gonna work on an assignment tomorrow, i need the fucking report asap by eod
detoxify > formalize: I am going to work on an assignment tomorrow. I need the report by Eod as soon as possible.
formalize > detoxify: I'm going to work on an assignment tomorrow. I need the report by Eod.
orders agree        : False

INPUT               : Who the hell sent the boxes to 47 Willow Lane? They were supposed to go to 92 Cedar Court.
detoxify > formalize: Who sent the boxes to 47 Willow Lane? They were supposed to go to 92 Cedar Court.
formalize > detoxify: Who sent the boxes to 47 Willow Lane? They were supposed to go to 92 Cedar Court.
orders agree        : True

INPUT               : Stop being such a jerk to Nina. She's covering 3 shifts this week because you bailed.
detoxify > formalize: Stop being rude to Nina because she is covering three shifts this week because you bailed.
formalize > detoxify: Stop being rude to Nina. She is covering three shifts this week because

## 8. Probe the limits
Both training corpora are **sentence-level** - GYAFC's longest example is 58 tokens, and ParaDetox comments are short too - so the model has no training signal for long passages whatever `MAX_INPUT_TOKENS` is set to. Both corpora are also written internet text, so unpunctuated voice-typed input is unfamiliar in a second way.

This cell makes both failure modes visible rather than leaving them to guesswork:

1. a normal short message - the case the model was trained for
2. a long message that still fits under the 128-token cap, so nothing is truncated going in; watch whether content goes missing coming **out**
3. the same long message with punctuation stripped, simulating voice typing
4. a message over the cap, showing that oversized input is **rejected with an error** rather than quietly cut in half

Token counts are printed alongside, because content loss is easier to see as a number than to spot by reading.

In [23]:
import re

short_message = "hey can u send me the deck b4 the mtg, kinda urgent"
long_message = (
    "Hey so i wanted to flag that the deployment yesterday went pretty badly and honestly "
    "nobody on the team had any clue what was going on, the staging env was totally broken "
    "for like three hours and we lost a bunch of test data, plus the rollback script didnt "
    "even work so we had to do it all by hand which was a nightmare, anyway can we please "
    "talk about this in standup tomorrow because it cant happen again"
)
unpunctuated = re.sub(r"[^\w\s]", "", long_message).lower()
oversized = long_message + " " + long_message  # Comfortably past MAX_INPUT_TOKENS.

for label, text in [
    ("1. short (expected case)", short_message),
    ("2. long, punctuated", long_message),
    ("3. long, no punctuation", unpunctuated),
    ("4. over the token cap", oversized),
]:
    print("=" * 70)
    print(label)
    input_tokens = tokenizer(FORMAL_PREFIX + text, truncation=False)["input_ids"]
    try:
        rewritten = formalize(text)
    except ValueError as error:
        print(f"  input tokens: {len(input_tokens)}  -> rejected: {error}")
        continue
    output_tokens = tokenizer(rewritten, truncation=False)["input_ids"]
    print(f"  input  ({len(input_tokens):>3} tokens): {text}")
    print(f"  output ({len(output_tokens):>3} tokens): {rewritten}")
    print(f"  output/input token ratio: {len(output_tokens) / len(input_tokens):.2f}")
    print("  (a ratio well below 1.0 on a long input means content was dropped)")

1. short (expected case)
  input  ( 25 tokens): hey can u send me the deck b4 the mtg, kinda urgent
  output ( 21 tokens): Hello, can you send me the deck with the Mtg? It is kind of urgent.
  output/input token ratio: 0.84
  (a ratio well below 1.0 on a long input means content was dropped)
2. long, punctuated
  input  ( 95 tokens): Hey so i wanted to flag that the deployment yesterday went pretty badly and honestly nobody on the team had any clue what was going on, the staging env was totally broken for like three hours and we lost a bunch of test data, plus the rollback script didnt even work so we had to do it all by hand which was a nightmare, anyway can we please talk about this in standup tomorrow because it cant happen again
  output ( 86 tokens): I wanted to flag that the deployment yesterday went pretty badly and honestly nobody on the team had any clue what was going on. The staging env was broken for three hours and we lost a bunch of test data. The rollback script didn't e

## 9. Type your own
Edit the string below and run the cell. Rerun **only this cell** for another sentence.

Alternatively, call `detoxify("...")`, `formalize("...")`, or `rewrite(PREFIX, "...")` in a new cell.

Review each result: the model may leave hostile wording in place, change meaning, or correctly leave an already-clean sentence unchanged.

In [24]:
sentence = "i dont think thats gonna work tbh"  # input("Enter a sentence: ")
if not sentence.strip():
    print("No sentence entered. Rerun this cell to try again.")
else:
    try:
        print("Original  :", sentence)
        print("detoxify: :", detoxify(sentence))
        print("formalize::", formalize(sentence))
        print("both      :", formalize(detoxify(sentence)))
    except ValueError as error:
        print(error)

Original  : i dont think thats gonna work tbh
detoxify: : i dont think thats gonna work tbh
formalize:: I do not think that will work.
both      : I do not think that will work.
